# Digits STDP classifier — end-to-end on the SpikeEngine STDP FPGA

Reproduces the ORNL ICONS digit-classifier tutorial on the wide-fixed-point
Basys3 STDP build: **on-chip STDP training** (bit-exact vs SuperNeuroMAT),
weight normalization, then **on-chip rate-readout inference**.

Accuracy note: the tutorial's headline "93%" is the one-vs-rest `(TP+TN)/total`
metric with a multi-answer decode, computed WITHOUT resetting membrane state
between test images. With a full reset per image (this notebook's methodology)
the same metric on the full 899-image test set is **one-vs-rest 0.909**
(recall 0.776) — an exact match to the software reference computed the same
way. On strict single-answer top-1 the same network is **58.4%**. On-chip
**training** is bit-exact vs SuperNeuroMAT (0/640 weights); on-chip
**inference** is ALSO bit-exact (an earlier apparent ~1-count drift was
traced to a host-side weight-loading bug, not a hardware limitation).

## 0. Configuration

In [ ]:
RUN_HARDWARE = True      # False -> software reference only
BOARD = 'basys3'      # 'basys3' | 'sp701' | 'zcu104'
PORT  = 'auto'           # 'auto' autodetects, or e.g. 'COM17'
NUM_TRAIN = 898          # full training pool (50% split)
NUM_TEST  = 250          # test images to score (full is 899; slow over UART)

# wide fixed-point: 16-bit weights, 24-bit membrane, frac_bits=11 (so the fine
# aneg=-0.0005 STDP tap survives per-update quantization -- frac=10 loses it).
FRAC_BITS, WEIGHT_W, DATA_W = 11, 16, 24

## 1. Build the training network + schedule

64 pixel inputs (threshold 1) fully connected to 10 output-class neurons
(**threshold 99** during training, so only the teacher fires them — this is the
key to avoiding weight saturation). Grayscale-amplitude input encoding, teacher
amplitude 100, and a **3-tap STDP rule with real depression**.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from superneuromat import spikeengine as se
from superneuromat.spikeengine.examples import digits_stdp_e2e as ex

digits = load_digits(n_class=10); input_max = digits.data.max()
Xtr, Xte, ytr, yte = train_test_split(digits.data, digits.target, test_size=0.5, shuffle=False)

# reuse the validated example helpers (same recipe as the .py module)
ex.FRAC_BITS = FRAC_BITS
net, ins, outs, schedule, total = ex.build_train_schedule(Xtr, ytr, NUM_TRAIN, input_max)
print(f'{len(net.pre_synaptic_neuron_ids)} synapses, {total} training ticks, {NUM_TRAIN} images')

## 2. Fixed-point-faithful software training reference

In [ ]:
sw_W = ex.software_train(Xtr, ytr, NUM_TRAIN, input_max)
print('software training reference done; io-weight range',
      round(float(sw_W[:64,64:74].min()),2), '..', round(float(sw_W[:64,64:74].max()),2))

## 3. Train on-chip and verify BIT-EXACT

`load_network` maps the fresh network onto the board; `run_schedule` replays the
training ticks (STDP learns on-chip). We then read every weight back off the chip
and confirm it matches the software reference exactly.

In [ ]:
if RUN_HARDWARE:
    dev = se.connect(port=PORT, board=BOARD)
    dev.soft_reset()
    info = se.load_network(dev, net, frac_bits=FRAC_BITS, weight_w=WEIGHT_W,
                           data_w=DATA_W, stdp_window=5)

    # A long silent cell (no stdout for the ~1-2 minutes this takes) is the
    # suspected trigger for a real hang seen running this notebook via nbconvert
    # against real hardware (17+ minutes, near-zero CPU, had to be killed --
    # short cells with frequent output ran fine). Printing periodic progress
    # keeps the kernel producing steady iopub traffic during the run.
    def _train_progress(ticks):
        ticks = list(ticks)
        n = len(ticks)
        for i, t in enumerate(ticks):
            if i % 200 == 0 or i == n - 1:
                print(f'  training tick {i}/{n}', flush=True)
            yield t

    se.run_schedule(dev, schedule, total, frac_bits=FRAC_BITS, n_neurons=74,
                    progress=_train_progress)
    hw_w = se.read_weights(dev, info['entry_index'], frac_bits=FRAC_BITS)
    mism = sum(1 for (d,s),v in hw_w.items() if v != sw_W[s,d])
    print(f'on-chip training: {mism}/640 weight mismatches ->',
          'BIT-EXACT vs SuperNeuroMAT' if mism==0 else 'MISMATCH')
else:
    print('hardware training skipped')

## 4. Normalize + load the inference network

Apply the tutorial's `+=5; *=0.5` normalization and switch to the inference
config (leak=0, output threshold 35) for the rate readout.

In [ ]:
inf, iin, iout = ex.build_infer_snn(sw_W)
if RUN_HARDWARE:
    dev.soft_reset()
    se.load_network(dev, inf, frac_bits=FRAC_BITS, weight_w=WEIGHT_W, data_w=DATA_W, stdp_window=5)
    dev.set_stdp_enable(False)
    print('inference network loaded')

## 5. On-chip rate-readout inference + accuracy

Each test image is presented as 20 spikes; the class whose output neuron fires
most (tie-broken by earliest spike) is the prediction.

In [ ]:
# one-vs-rest metric, full reset per image; scores NUM_TEST images (set 899 for
# the full-set headline: one-vs-rest 0.909, top-1 58.4%)
Xt, yt = Xte[:NUM_TEST], yte[:NUM_TEST]
sw_acc, sw_rec = ex.one_vs_rest(lambda im: ex.software_infer_answers(inf, iin, iout, im, input_max), Xt, yt)
print(f'software (frac=11 weights): one-vs-rest acc={sw_acc:.3f}  recall={sw_rec:.3f}')
if RUN_HARDWARE:
    # Progress printing every 25 images, same reasoning as the training cell's
    # _train_progress -- this loop is ~500 lock-step hardware round trips
    # (2 passes x NUM_TEST images), several minutes of otherwise-silent cell.
    tp = tn = fp = fn = 0
    top1 = 0
    for k in range(len(Xt)):
        if k % 25 == 0 or k == len(Xt) - 1:
            print(f'  on-chip inference: image {k}/{len(Xt)}', flush=True)
        ans = ex.hardware_infer_answers(dev, iin, iout, Xt[k], input_max)
        real = int(yt[k])
        for lab in range(10):
            tp += (lab == real) and (lab in ans); fn += (lab == real) and (lab not in ans)
            tn += (lab != real) and (lab not in ans); fp += (lab != real) and (lab in ans)
        pred = ex.hardware_infer(dev, iin, iout, Xt[k], input_max)
        top1 += int(pred == real)
    hw_acc = (tp + tn) / (tp + tn + fp + fn)
    hw_rec = tp / (tp + fn) if (tp + fn) else 0.0
    print(f'FPGA (on-chip):             one-vs-rest acc={hw_acc:.3f}  recall={hw_rec:.3f}')
    print(f'FPGA strict single-answer top-1: {top1}/{len(Xt)} = {top1/len(Xt)*100:.1f}%')

## 6. Single-image LED demo

Pick one test digit, run it, and see which output neuron fires — the board's
LED[class] lights up (SPIKE_MON_BASE=64 maps LED[0..9] to the class neurons).

In [ ]:
import random

idx = random.randrange(len(Xte))
true = int(yte[idx])
if RUN_HARDWARE:
    pred = ex.hardware_infer(dev, iin, iout, Xte[idx], input_max)
    print(f'test digit #{idx}: true={true} -> LED[{pred}] lights',
          '(correct)' if pred==true else '(wrong)')

## 7. Cleanup

In [ ]:
if RUN_HARDWARE:
    dev.close()